In [1]:
import os
import numpy as np
import pandas as pd
from PIL import Image
import librosa
import soundfile as sf

# ---------- IMAGE EXTRACTION ----------
def extract_image_data(image_path):
    with Image.open(image_path) as img:
        img_arr = np.array(img)

        return {
            "file_name": os.path.basename(image_path),
            "width": img.size[0],
            "height": img.size[1],
            "mode": img.mode,
            "file_size_kb": round(os.path.getsize(image_path)/1024, 2),
            "mean_pixel": float(np.mean(img_arr)),
            "std_pixel": float(np.std(img_arr))
        }

# ---------- AUDIO EXTRACTION ----------
def extract_audio_data(audio_path):
    y, sr = librosa.load(audio_path, sr=None)

    data = {
        "file_name": os.path.basename(audio_path),
        "duration_sec": round(librosa.get_duration(y=y, sr=sr), 2),
        "sample_rate": sr,
        "file_size_kb": round(os.path.getsize(audio_path)/1024, 2),
        "rms_energy": float(librosa.feature.rms(y=y).mean()),
        "zero_crossing_rate": float(librosa.feature.zero_crossing_rate(y).mean())
    }

    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    for i in range(13):
        data[f"mfcc_{i+1}_mean"] = float(np.mean(mfcc[i]))

    if audio_path.endswith(".wav"):
        info = sf.info(audio_path)
        data["channels"] = info.channels
        data["bit_depth"] = info.subtype

    return data

# ---------- EXTRACT FROM CURRENT DIRECTORY ----------
def extract_from_current_directory():
    image_data = []
    audio_data = []

    for file in os.listdir(os.getcwd()):
        if file.lower().endswith((".jpg", ".jpeg", ".png")):
            image_data.append(extract_image_data(file))

        elif file.lower().endswith((".mp3", ".wav")):
            audio_data.append(extract_audio_data(file))

    return pd.DataFrame(image_data), pd.DataFrame(audio_data)


# ---------- RUN ----------
img_df, aud_df = extract_from_current_directory()

img_df.to_csv("image_features.csv", index=False)
aud_df.to_csv("audio_features.csv", index=False)

print("✅ Extraction completed from current directory")


✅ Extraction completed from current directory
